# 🤖 Apptronik Apollo Humanoid - Huấn Luyện RL (Google Colab GPU Suite)
### Giai đoạn 1: Stage 1 Standing Balance & Disturbance Recovery Policy (MJX + JAX PPO)
---
**Ưu điểm tối ưu cho Google Colab:**
- ✅ **Đồng bộ 100% với Kaggle:** Cùng thuật toán JAX + MuJoCo MJX (4096 môi trường song song cực nhanh).
- ✅ **Tự động lưu Google Drive:** Checkpoints được lưu trực tiếp vào Drive của bạn, không bao giờ sợ mất weights khi tắt Colab.
- ✅ **Cơ chế Tự Động Resume:** Khi chuyển từ Kaggle sang Colab, hệ thống tự động quét và nạp file `.npz` cũ để train nối tiếp từ bước đã đạt được!

In [ ]:
# 1. Kiểm tra GPU & Cấu hình phần cứng
!nvidia-smi
import os
print("[INFO] Thu muc lam viec:", os.getcwd())

In [ ]:
# 2. Cài đặt các thư viện lõi (MuJoCo MJX, JAX GPU, Flax, Optax)
!pip install -q --no-cache-dir mujoco mujoco-mjx flax optax
import jax
print("[INFO] JAX Backend:", jax.default_backend(), "| Thiet bi GPU:", jax.devices())
assert jax.default_backend() in ("gpu", "tpu"), "Vui long vao Runtime -> Change runtime type -> Chon GPU T4!"

In [ ]:
# 3. Kết nối Google Drive để lưu Checkpoints lâu dài (Tự động)
import os
USE_GDRIVE = True
GDRIVE_DIR = "/content/drive/MyDrive/Apollo_Checkpoints"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(GDRIVE_DIR, exist_ok=True)
    print(f"[OK] Da ket noi Google Drive! Checkpoints se luu tai: {GDRIVE_DIR}")
except Exception as e:
    print(f"[CANH BAO] Khong the mount Drive ({e}). Se luu tam tai /content/checkpoints/")
    USE_GDRIVE = False
    GDRIVE_DIR = "/content/checkpoints"
    os.makedirs(GDRIVE_DIR, exist_ok=True)

In [ ]:
# 4. Tải mô hình Robot Apptronik Apollo 3D
import os
if not os.path.exists("mujoco_menagerie"):
    os.system("git clone --depth 1 https://github.com/google-deepmind/mujoco_menagerie.git")
assert os.path.exists("mujoco_menagerie/apptronik_apollo/scene.xml"), "Loi: Khong tim thay file scene.xml!"
print("[OK] Mo hinh Robot Apollo 3D da san sang.")

In [ ]:
# 5. Huấn luyện Reinforcement Learning (PPO MJX 4096 Envs + Auto-Resume)
import os, time, math, glob
import jax, jax.numpy as jnp
import optax, flax, flax.linen as nn
import mujoco, flax.traverse_util
from mujoco import mjx

print("=" * 64)
print("  APOLLO HUMANOID - GOOGLE COLAB TRAINING PIPELINE (v13)")
print("  Moi truong: 4096 Envs song song | Thuat toan: PPO MJX")
print("=" * 64)

# --- 1. Mạng Actor-Critic ---
class ActorCritic(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, obs):
        x = obs
        for h in (512, 256, 128):
            x = nn.elu(nn.Dense(h)(x))
        mean = nn.tanh(nn.Dense(self.action_dim)(x))
        log_std = self.param("log_std", nn.initializers.constant(-0.5), (self.action_dim,))
        log_std = jnp.clip(log_std, -3.0, 0.5)
        value = nn.Dense(1)(x).squeeze(-1)
        return mean, log_std, value

# --- 2. Mô hình Vật Lý MJX ---
model_path = "mujoco_menagerie/apptronik_apollo/scene.xml"
mj_model = mujoco.MjModel.from_xml_path(model_path)
SIM_DT, N_SUBSTEPS = 0.002, 5
CTRL_DT = SIM_DT * N_SUBSTEPS  # 0.01s

mj_model.opt.timestep = SIM_DT
mj_model.opt.iterations = 4
mj_model.opt.ls_iterations = 4
mj_model.opt.integrator = mujoco.mjtIntegrator.mjINT_IMPLICITFAST
for i in range(mj_model.ngeom):
    mj_model.geom_solref[i, 0] = 0.004
    mj_model.geom_solref[i, 1] = 1.0
    mj_model.geom_solimp[i, :] = [0.9, 0.95, 0.001, 0.5, 2.0]

mjx_model = mjx.put_model(mj_model)
nq, nv, nu = mj_model.nq, mj_model.nv, mj_model.nu
ctrl_range = jnp.array(mj_model.actuator_ctrlrange)

key_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_KEY, "stand")
if key_id < 0: key_id = 0
default_qpos = jnp.array(mj_model.key_qpos[key_id])
default_ctrl = jnp.array(mj_model.key_ctrl[key_id])
default_pose = jnp.array(mj_model.key_qpos[key_id][7:])

Z_NOMINAL = float(default_qpos[2])
TRACKING_SIGMA = 0.25
ACTION_SCALE = 0.3
EPISODE_LEN = 1000
TERM_HEIGHT = Z_NOMINAL * 0.75
TERM_TILT = 0.5
OBS_DIM = 3 + 3 + 3 + nu + nu + nu

def get_upvector(qpos):
    qw, qx, qy, qz = qpos[3], qpos[4], qpos[5], qpos[6]
    return jnp.array([
        2.0 * (qx*qz + qw*qy),
        2.0 * (qy*qz - qw*qx),
        1.0 - 2.0 * (qx**2 + qy**2),
    ])

def get_obs(d, prev_act):
    upvec = get_upvector(d.qpos)
    linvel = d.qvel[:3]
    angvel = d.qvel[3:6]
    jpos = d.qpos[7:7+nu] - default_pose
    jvel = d.qvel[6:6+nu]
    obs = jnp.concatenate([upvec, linvel, angvel, jpos, jvel, prev_act])
    return jnp.clip(obs, -20.0, 20.0)

def compute_reward(qpos, qvel, torques, action, prev_action):
    upvec = get_upvector(qpos)
    tracking_lin_vel = jnp.exp(-jnp.sum(jnp.square(qvel[:2])) / TRACKING_SIGMA)
    tracking_ang_vel = jnp.exp(-jnp.square(qvel[5]) / TRACKING_SIGMA)
    cost_linvel_z = jnp.square(qvel[2])
    cost_angvel_xy = jnp.sum(jnp.square(qvel[3:5]))
    cost_orientation = jnp.sum(jnp.square(upvec[:2]))
    jpos = qpos[7:7+nu]
    cost_stand_still = jnp.sum(jnp.abs(jpos - default_pose))
    cost_torques = (jnp.sqrt(jnp.sum(jnp.square(torques))) + jnp.sum(jnp.abs(torques)))
    cost_action_rate = jnp.sum(jnp.square(action - prev_action))
    out_lo = -jnp.clip(jpos - ctrl_range[:nu, 0], -jnp.inf, 0.0)
    out_hi = jnp.clip(jpos - ctrl_range[:nu, 1], 0.0, jnp.inf)
    cost_dof_limits = jnp.sum(out_lo + out_hi)
    total = (
        1.0 * tracking_lin_vel
        + 0.5 * tracking_ang_vel
        - 2.0 * cost_linvel_z
        - 0.05 * cost_angvel_xy
        - 1.0 * cost_orientation
        - 0.5 * cost_stand_still
        - 1e-4 * cost_torques
        - 0.01 * cost_action_rate
        - 10.0 * cost_dof_limits
    )
    return total * CTRL_DT

def env_reset(rng):
    rng_q, rng_v, rng_j = jax.random.split(rng, 3)
    noise = jax.random.uniform(rng_j, (nq-7,), minval=0.85, maxval=1.15)
    qpos = jnp.concatenate([
        default_qpos[:7] + jax.random.uniform(rng_q, (7,), minval=-0.01, maxval=0.01),
        default_qpos[7:] * noise
    ])
    dv = jax.random.uniform(rng_v, (nv,), minval=-0.05, maxval=0.05)
    d = mjx.make_data(mjx_model)
    d = d.replace(qpos=qpos, qvel=dv)
    d = mjx.forward(mjx_model, d)
    prev_act = jnp.zeros(nu)
    return {"d": d, "prev_act": prev_act, "step": jnp.zeros((), jnp.int32)}

def env_step(state, action_and_rng):
    raw_act, rng_reset = action_and_rng
    d, prev_act, step = state["d"], state["prev_act"], state["step"]
    ctrl = jnp.clip(default_ctrl + raw_act * ACTION_SCALE, ctrl_range[:,0], ctrl_range[:,1])
    d = d.replace(ctrl=ctrl)
    def _sub(dd, _): return mjx.step(mjx_model, dd), None
    d, _ = jax.lax.scan(_sub, d, None, length=N_SUBSTEPS)
    torques = d.qfrc_actuator[6:6+nu]
    rew = compute_reward(d.qpos, d.qvel, torques, raw_act, prev_act)
    obs_out = get_obs(d, raw_act)
    upvec = get_upvector(d.qpos)
    terminated = jnp.logical_or(upvec[2] < TERM_TILT, d.qpos[2] < TERM_HEIGHT)
    step_new = step + 1
    truncated = step_new >= EPISODE_LEN
    done = jnp.logical_or(terminated, truncated)
    reset_state = env_reset(rng_reset)
    next_d = jax.tree.map(lambda r, c: jnp.where(done, r, c), reset_state["d"], d)
    next_act = jnp.where(done, jnp.zeros(nu), raw_act)
    next_step = jnp.where(done, jnp.zeros((), jnp.int32), step_new)
    nst = {"d": next_d, "prev_act": next_act, "step": next_step}
    return obs_out, nst, rew, terminated, truncated

# --- 3. Cấu hình PPO ---
NUM_ENVS = 4096
ROLLOUT = 32
GAMMA = 0.99
LAM = 0.95
CLIP_EPS = 0.2
ENT_COEF = 0.01
VF_COEF = 0.5
MAX_GRAD = 0.5
TOTAL_STEPS = 100_000_000
STEPS_PER_IT = NUM_ENVS * ROLLOUT
N_ITERS = TOTAL_STEPS // STEPS_PER_IT

lr_schedule = optax.linear_schedule(3e-4, 3e-5, N_ITERS)
network = ActorCritic(action_dim=nu)
rng = jax.random.PRNGKey(42)
rng, ri = jax.random.split(rng)
params = network.init(ri, jnp.zeros((1, OBS_DIM)))
tx = optax.chain(optax.clip_by_global_norm(MAX_GRAD), optax.adam(lr_schedule, eps=1e-5))
opt_state = tx.init(params)

# --- 4. Tự Động Kiểm Tra & Nạp Checkpoint Cũ (Resume) ---
existing_checkpoints = sorted(glob.glob(f"{GDRIVE_DIR}/apollo_stage1_v13_step_*.npz"))
if not existing_checkpoints:
    existing_checkpoints = sorted(glob.glob("/content/checkpoints/apollo_stage1_v13_step_*.npz"))

start_iter = 1
cur_steps = 0

if existing_checkpoints:
    latest_ck = existing_checkpoints[-1]
    print(f"[RESUME] Phat hien checkpoint truoc do: {latest_ck}")
    try:
        loaded_data = jnp.load(latest_ck)
        flat_params = {k: loaded_data[k] for k in loaded_data.files}
        params = flax.traverse_util.unflatten_dict(flat_params, sep="/")
        step_str = latest_ck.split("_")[-1].replace(".npz", "")
        if step_str.isdigit():
            cur_steps = int(step_str)
            start_iter = cur_steps // STEPS_PER_IT + 1
        print(f"--> [THANH CONG] Da tiep tuc tu buoc {cur_steps:,} (Iteration {start_iter}/{N_ITERS})!")
    except Exception as e:
        print(f"--> [LOI] Khong the nap checkpoint: {e}. Se bat dau tu dau!")
else:
    print("[INIT] Khong co checkpoint cu, bat dau huan luyen moi tu Step 0!")

# Khởi tạo 4096 môi trường
rng_envs = jax.random.split(rng, NUM_ENVS)
states = jax.vmap(env_reset)(rng_envs)

@jax.jit
def ppo_iter(params, opt_state, states, rng):
    def _step(carry, _):
        st, p, r = carry
        r, ra, r_reset = jax.random.split(r, 3)
        r_resets = jax.random.split(r_reset, NUM_ENVS)
        obs = jax.vmap(lambda s: get_obs(s["d"], s["prev_act"]))(st)
        mu, ls, val = network.apply(p, obs)
        std = jnp.exp(ls)
        act = jnp.clip(mu + std * jax.random.normal(ra, mu.shape), -1., 1.)
        lp = -0.5 * jnp.sum(jnp.square((act - mu) / (std + 1e-8)) + 2.0 * ls + math.log(2.0 * math.pi), axis=-1)
        lp = jnp.clip(lp, -10.0, 10.0)
        _, nst, rew, term, trunc = jax.vmap(env_step)(st, (act, r_resets))
        return (nst, p, r), (obs, act, lp, val, rew, term, trunc)

    (fst, _, rng), traj = jax.lax.scan(_step, (states, params, rng), None, length=ROLLOUT)
    obs, act, old_lp, vals, rews, terms, truncs = traj
    lobs = jax.vmap(lambda s: get_obs(s["d"], s["prev_act"]))(fst)
    _, _, nv_last = network.apply(params, lobs)

    def _gae(carry, t):
        gae, nxv = carry
        r, v, term, trunc = rews[t], vals[t], terms[t], truncs[t]
        done = jnp.logical_or(term, trunc)
        delta = r + GAMMA * nxv * (1. - term.astype(jnp.float32)) - v
        gae = delta + GAMMA * LAM * (1. - done.astype(jnp.float32)) * gae
        return (gae, v), gae

    _, advs = jax.lax.scan(_gae, (jnp.zeros(NUM_ENVS), nv_last), jnp.arange(ROLLOUT - 1, -1, -1))
    advs = jnp.flip(advs, axis=0)
    rets = advs + vals
    advs = (advs - advs.mean()) / (advs.std() + 1e-8)

    flat = lambda x: x.reshape(-1, *x.shape2:]) if hasattr(x, "shape2") else x.reshape(-1, *x.shape[2:])
    fo, fa, flp, fadv, fret = map(flat, [obs, act, old_lp, advs, rets])
    ovf = flat(vals)

    def loss_fn(p):
        mu, ls, v = network.apply(p, fo)
        std = jnp.exp(ls)
        lp = -0.5 * jnp.sum(jnp.square((fa - mu) / (std + 1e-8)) + 2.0 * ls + math.log(2.0 * math.pi), axis=-1)
        lp = jnp.clip(lp, -10.0, 10.0)
        lr_ = jnp.clip(lp - flp, -5.0, 5.0)
        ratio = jnp.exp(lr_)
        pg = -jnp.mean(jnp.minimum(ratio * fadv, jnp.clip(ratio, 1-CLIP_EPS, 1+CLIP_EPS)*fadv))
        vc = ovf + jnp.clip(v - ovf, -5.0, 5.0)
        vf = VF_COEF * jnp.mean(jnp.maximum(jnp.square(v-fret), jnp.square(vc-fret)))
        ent = -ENT_COEF * jnp.mean(jnp.sum(ls + 0.5*math.log(2*math.pi*math.e), axis=-1))
        total = pg + vf + ent
        return jnp.where(jnp.isnan(total), 0.0, total)

    loss, grads = jax.value_and_grad(loss_fn)(params)
    grads = jax.tree.map(lambda g: jnp.where(jnp.isnan(g), 0.0, g), grads)
    upd, opt_state = tx.update(grads, opt_state, params)
    params = optax.apply_updates(params, upd)
    return params, opt_state, fst, rng, jnp.mean(rews), loss

# --- 5. Vòng Lặp Huấn Luyện Chính ---
t0 = time.time()
print(f"\n>>> BAT DAU HUAN LUYEN TU ITERATION {start_iter} DEN {N_ITERS}...")

for it in range(start_iter, N_ITERS + 1):
    t1 = time.time()
    params, opt_state, states, rng, mr, loss = ppo_iter(params, opt_state, states, rng)
    jax.block_until_ready(params)
    cur_steps += STEPS_PER_IT
    sps = STEPS_PER_IT / max(1e-5, time.time() - t1)

    if it % 10 == 0 or it == start_iter:
        r_val = float(mr)
        if r_val > 0.005: status = "*** DA DUNG THANG CAN BANG (STANDING) ***"
        elif r_val > -0.01: status = "Sap dung thang (Near-standing)"
        elif r_val > -0.03: status = "Dang tien bo (Improving)"
        else: status = "Dang kham pha (Exploring)"
        print(f"[{it:04d}/{N_ITERS}] steps={cur_steps:,} | " 
              f"rew={r_val:.5f} | loss={float(loss):.4f} | " 
              f"sps={sps:,.0f} | time={time.time()-t0:.0f}s -> {status}", flush=True)

    # Lưu Checkpoint mỗi 50 iterations sang cả Local và Google Drive
    if it % 50 == 0 or it == N_ITERS:
        flat_dict = flax.traverse_util.flatten_dict(params, sep="/")
        ck_local = f"/content/checkpoints/apollo_stage1_v13_step_{cur_steps}.npz"
        os.makedirs("/content/checkpoints", exist_ok=True)
        jnp.savez(ck_local, **flat_dict)
        
        if USE_GDRIVE:
            ck_drive = f"{GDRIVE_DIR}/apollo_stage1_v13_step_{cur_steps}.npz"
            jnp.savez(ck_drive, **flat_dict)
            print(f"  -> [DA LUU GOOGLE DRIVE]: {ck_drive}", flush=True)
        else:
            print(f"  -> [DA LUU LOCAL]: {ck_local}", flush=True)

print("\n" + "=" * 64)
print("  [HOAN THANH HUAN LUYEN APOLLO STAGE 1 V13 TREN GOOGLE COLAB]")
print("=" * 64)